# SIH NB4 CV Pilot

Attach NB2 full saved output. Internet ON, Accelerator None. No credentials or original SIH dataset needed.

Run the three-source smoke check before the 100-source batch. This is a single-scene download and alignment pilot, not seasonal composites or a training dataset. India is excluded. Unmatched sources remain unlabelled. SCL/cloud filtering can introduce availability bias; inspect failures by country.

Sources: [Earth Search](https://github.com/Element84/earth-search), [ESA WorldCover](https://esa-worldcover.org/en/data-access), [Rasterio](https://rasterio.readthedocs.io/en/stable/api/rasterio.vrt.html).


In [ ]:
%pip install -q rasterio==1.4.4 pyarrow requests matplotlib


In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import math
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import rasterio
from rasterio.enums import Resampling
from rasterio.errors import RasterioError
from rasterio.transform import from_origin
from rasterio.vrt import WarpedVRT
from rasterio.warp import transform, transform_bounds

ROOT = Path("/kaggle/working/cv_pilot_v1")
ROOT.mkdir(parents=True, exist_ok=True)
COUNTRIES = ["Algeria", "Angola", "Indonesia", "Iraq", "Libya", "Nigeria"]
BANDS = ["blue", "green", "red", "nir", "swir16", "swir22"]
SIZE = 200
PIXEL_M = 10
MAX_SCENES = 8
MIN_CLEAR = 0.80
DATES = "2022-01-01T00:00:00Z/2024-12-31T23:59:59Z"
STAC = "https://earth-search.aws.element84.com/v1/search"
COLLECTION = "sentinel-2-c1-l2a"
WC_BASE = "https://esa-worldcover.s3.eu-central-1.amazonaws.com/"
SEED = 42

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "POST"],
)))
print("Output:", ROOT)
print("No credentials are needed.")


In [ ]:
columns = ["source_id", "block_id", "lat", "lon",
           "is_eog_flare", "eog_flare_id"]
parts = []
input_files = {}

for index, country in enumerate(COUNTRIES):
    name = f"features_{country}_2022_2024.parquet"
    matches = sorted(Path("/kaggle/input").rglob(name))
    if len(matches) != 1:
        raise RuntimeError(
            f"{country}: expected one {name}, found {matches}. "
            "Attach NB2 full output, not only its results ZIP."
        )
    input_files[country] = str(matches[0])
    frame = pd.read_parquet(matches[0], columns=columns)
    if frame["source_id"].isna().any() or frame["source_id"].duplicated().any():
        raise ValueError(f"{country}: missing or duplicate source_id")
    if frame["block_id"].isna().any():
        raise ValueError(f"{country}: missing block_id")
    if not frame["is_eog_flare"].isin([0, 1]).all():
        raise ValueError(f"{country}: invalid labels")
    if not (np.isfinite(frame[["lat", "lon"]]).all().all()
            and frame.lat.between(-80, 84).all()
            and frame.lon.between(-180, 180, inclusive="left").all()):
        raise ValueError(f"{country}: invalid coordinates")
    frame["source_id"] = frame["source_id"].astype(str)
    frame["country"] = country
    frame = frame.sample(frac=1, random_state=SEED + index)

    # One positive source per known site and one sampled source per spatial block.
    positives = frame[frame.is_eog_flare.eq(1)].copy()
    if positives.eog_flare_id.isna().any():
        raise ValueError(f"{country}: positive without EOG site ID")
    positives = positives.drop_duplicates("eog_flare_id")
    positives = positives.drop_duplicates("block_id").head(8)
    quota = 17 if index < 4 else 16
    unlabeled = frame[
        frame.is_eog_flare.eq(0)
        & ~frame.block_id.isin(positives.block_id)
    ].drop_duplicates("block_id").head(quota - len(positives))
    chosen = pd.concat([positives, unlabeled], ignore_index=True)
    if len(chosen) != quota:
        raise ValueError(f"{country}: insufficient independent pilot sources")
    parts.append(chosen)

pilot = pd.concat(parts, ignore_index=True)
assert len(pilot) == 100
assert "India" not in set(pilot.country)
assert not pilot.duplicated(["country", "block_id"]).any()
pilot["chip_id"] = [
    hashlib.sha256(f"{c}:{s}".encode()).hexdigest()[:20]
    for c, s in zip(pilot.country, pilot.source_id)
]
assert pilot.chip_id.is_unique
pilot = pilot.sort_values(["country", "chip_id"]).reset_index(drop=True)

config = {
    "version": 1, "seed": SEED, "countries": COUNTRIES,
    "sentinel_collection": COLLECTION,
    "bands": BANDS, "shape": [6, SIZE, SIZE], "pixel_m": PIXEL_M,
    "dates": DATES, "max_scenes": MAX_SCENES, "minimum_clear_fraction": MIN_CLEAR,
    "clear_scl_classes": [4, 5, 6], "holdout_loaded": False,
    "purpose": "download and alignment pilot, not a training/evaluation dataset",
    "sample_sha256": hashlib.sha256(pilot.to_csv(index=False).encode()).hexdigest(),
    "input_files": input_files,
    "versions": {p: importlib.metadata.version(p)
                 for p in ["rasterio", "numpy", "pandas", "requests"]},
}
config_path = ROOT / "run_config.json"
if config_path.exists() and json.loads(config_path.read_text()) != config:
    raise RuntimeError("Existing pilot has a different configuration. Use a new ROOT.")
config_path.write_text(json.dumps(config, indent=2), encoding="utf-8")
pilot.to_csv(ROOT / "pilot_sources.csv", index=False)
display(pilot.groupby("country").agg(
    sources=("source_id", "size"), eog_positive=("is_eog_flare", "sum")
))


In [ ]:
def request_json(url, payload):
    response = session.post(url, json=payload, timeout=(15, 60))
    response.raise_for_status()
    return response.json()


def load_worldcover_keys():
    cache = ROOT / "worldcover_keys.json"
    if cache.exists():
        return set(json.loads(cache.read_text()))
    keys = set()
    params = {"list-type": "2", "prefix": "v200/2021/map/", "max-keys": 1000}
    ns = {"s": "http://s3.amazonaws.com/doc/2006-03-01/"}
    while True:
        response = session.get(WC_BASE, params=params, timeout=(15, 60))
        response.raise_for_status()
        xml = ET.fromstring(response.content)
        keys.update(node.text for node in xml.findall(".//s:Contents/s:Key", ns))
        if xml.findtext("s:IsTruncated", namespaces=ns) != "true":
            break
        token = xml.findtext("s:NextContinuationToken", namespaces=ns)
        if not token:
            raise ValueError("WorldCover listing omitted its next-page token")
        params["continuation-token"] = token
    if not keys:
        raise ValueError("WorldCover returned no tile keys")
    cache.write_text(json.dumps(sorted(keys)), encoding="utf-8")
    return keys


def crop_grid(lon, lat):
    zone = min(60, int((lon + 180) // 6) + 1)
    epsg = (32600 if lat >= 0 else 32700) + zone
    crs = f"EPSG:{epsg}"
    xs, ys = transform("EPSG:4326", crs, [lon], [lat])
    half = SIZE * PIXEL_M / 2
    affine = from_origin(xs[0] - half, ys[0] + half, PIXEL_M, PIXEL_M)
    bounds = (xs[0] - half, ys[0] - half, xs[0] + half, ys[0] + half)
    return crs, affine, transform_bounds(crs, "EPSG:4326", *bounds)


def read_crop(url, crs, affine, resampling=Resampling.nearest):
    if not url.startswith("https://"):
        raise ValueError(f"Not a public HTTPS asset: {url}")
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY="3", GDAL_HTTP_RETRY_DELAY="1",
        GDAL_HTTP_CONNECTTIMEOUT="15", GDAL_HTTP_TIMEOUT="60",
    ):
        with rasterio.open(url) as source:
            with WarpedVRT(
                source, crs=crs, transform=affine, width=SIZE, height=SIZE,
                src_nodata=source.nodata if source.nodata is not None else 0,
                nodata=np.nan, dtype="float32", resampling=resampling,
            ) as warped:
                return warped.read(1)


def worldcover_crop(bounds, crs, affine, available):
    west, south, east, north = bounds
    if east - west > 1:
        raise ValueError("Antimeridian crop requires separate handling")
    output = np.zeros((SIZE, SIZE), dtype="uint8")
    used = []
    for lat0 in range(3 * math.floor(south / 3), 3 * math.floor(north / 3) + 1, 3):
        for lon0 in range(3 * math.floor(west / 3), 3 * math.floor(east / 3) + 1, 3):
            tile = (f"{'N' if lat0 >= 0 else 'S'}{abs(lat0):02d}"
                    f"{'E' if lon0 >= 0 else 'W'}{abs(lon0):03d}")
            key = f"v200/2021/map/ESA_WorldCover_10m_2021_v200_{tile}_Map.tif"
            if key not in available:
                continue
            url = WC_BASE + key
            raster = read_crop(url, crs, affine)
            valid = np.isfinite(raster) & (raster != 0)
            output[valid] = raster[valid].astype("uint8")
            used.append(url)
    allowed = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
    if not np.isin(output, allowed).all():
        raise ValueError("Unexpected WorldCover class values")
    return output, used


def sentinel_crop(lon, lat, crs, affine):
    response = request_json(STAC, {
        "collections": [COLLECTION],
        "intersects": {"type": "Point", "coordinates": [lon, lat]},
        "datetime": DATES, "limit": MAX_SCENES,
        "sortby": [{"field": "properties.eo:cloud_cover", "direction": "asc"}],
    })
    attempts = []
    for item in response["features"][:MAX_SCENES]:
        try:
            assets = item["assets"]
            scl_float = read_crop(assets["scl"]["href"], crs, affine)
            scl = np.nan_to_num(scl_float, nan=0).astype("uint8")
            valid = np.isin(scl, [4, 5, 6])
            if valid.mean() < MIN_CLEAR or valid[75:125, 75:125].mean() < MIN_CLEAR:
                attempts.append({"item": item["id"], "reason": "cloud/coverage",
                                 "clear_fraction": float(valid.mean())})
                continue
            reflectance = []
            for band in BANDS:
                asset = assets[band]
                raw = read_crop(asset["href"], crs, affine, Resampling.bilinear)
                metadata = asset["raster:bands"][0]
                scale = float(metadata["scale"])
                offset = float(metadata["offset"])
                if not (np.isfinite(scale) and scale > 0 and np.isfinite(offset)):
                    raise ValueError(f"Invalid scale or offset: {band}")
                reflectance.append(raw * scale + offset)
            image = np.stack(reflectance).astype("float32")
            valid &= np.isfinite(image).all(axis=0)
            if valid.mean() < MIN_CLEAR or valid[75:125, 75:125].mean() < MIN_CLEAR:
                attempts.append({"item": item["id"], "reason": "missing band pixels"})
                continue
            if (image[:, valid] < -0.05).mean() > 0.05:
                raise ValueError("Suspect reflectance offset: many values below -0.05")
            image[:, ~valid] = np.nan
            return image, scl, valid, item, attempts
        except (requests.RequestException, RasterioError, KeyError, ValueError) as error:
            attempts.append({"item": item.get("id"), "reason": str(error)[:700]})
    raise RuntimeError(
        "No acceptable scene among the limited candidates: " + json.dumps(attempts)
    )


def validate_chip(path):
    with np.load(path, allow_pickle=False) as data:
        image = data["reflectance"]
        valid = data["valid_mask"]
        assert image.shape == (6, SIZE, SIZE) and image.dtype == np.float32
        assert valid.shape == (SIZE, SIZE) and valid.dtype == bool
        assert valid.mean() >= MIN_CLEAR
        assert np.isfinite(image[:, valid]).all()
        assert np.isnan(image[:, ~valid]).all()
        assert data["worldcover"].shape == valid.shape
        assert np.isin(data["worldcover"], [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]).all()


def run_pilot(rows):
    available = load_worldcover_keys()
    records = []
    for number, row in enumerate(rows.itertuples(index=False), start=1):
        metadata_path = ROOT / f"{row.chip_id}.json"
        chip_path = ROOT / f"{row.chip_id}.npz"
        if metadata_path.exists():
            previous = json.loads(metadata_path.read_text())
            if previous["status"] == "ok" and chip_path.exists():
                validate_chip(chip_path)
                records.append(previous)
                print(number, row.country, "cached", flush=True)
                continue

        record = {
            "source_id": row.source_id, "country": row.country,
            "chip_id": row.chip_id, "status": "failed",
            "chip_file": chip_path.name,
        }
        try:
            crs, affine, bounds = crop_grid(float(row.lon), float(row.lat))
            image, scl, valid, item, attempts = sentinel_crop(
                float(row.lon), float(row.lat), crs, affine
            )
            wc, wc_urls = worldcover_crop(bounds, crs, affine, available)
            # Zero means missing WorldCover, not a background/negative label.
            np.savez_compressed(
                chip_path, reflectance=image, scl=scl, valid_mask=valid,
                worldcover=wc, worldcover_valid=wc != 0,
                transform=np.asarray(tuple(affine)[:6]), crs=np.asarray(crs),
                bands=np.asarray(BANDS),
            )
            validate_chip(chip_path)
            record.update(
                status="ok", crs=crs, clear_fraction=float(valid.mean()),
                center_clear_fraction=float(valid[75:125, 75:125].mean()),
                worldcover_valid_fraction=float((wc != 0).mean()),
                reflectance_p01=float(np.nanpercentile(image, 1)),
                reflectance_p99=float(np.nanpercentile(image, 99)),
                scene_id=item["id"], scene_datetime=item["properties"]["datetime"],
                worldcover_urls=wc_urls, rejected_scenes=attempts, stac_item=item,
            )
        except (requests.RequestException, RasterioError, RuntimeError,
                KeyError, ValueError, AssertionError) as error:
            record["error"] = f"{type(error).__name__}: {error}"
        metadata_path.write_text(json.dumps(record, indent=2), encoding="utf-8")
        records.append(record)
        print(number, "/", len(rows), row.country, record["status"],
              record.get("error", "")[:180], flush=True)

    summary = pd.DataFrame([
        {key: value for key, value in record.items()
         if key not in {"stac_item", "rejected_scenes", "worldcover_urls"}}
        for record in records
    ])
    summary.to_csv(ROOT / "download_manifest.csv", index=False)
    return summary


In [ ]:
# Check one sampled source from each of three different countries first.
smoke_sources = pilot.groupby("country", sort=True).head(1).head(3)
smoke_results = run_pilot(smoke_sources)
display(smoke_results)
if not smoke_results.status.eq("ok").any():
    raise RuntimeError("All smoke downloads failed. Share the error column; do not scale up.")


In [ ]:
# Run only after reviewing the three-source smoke output.
download_results = run_pilot(pilot)
display(pd.crosstab(download_results.country, download_results.status))
display(download_results[
    ["country", "source_id", "status", "worldcover_valid_fraction"]
    if "worldcover_valid_fraction" in download_results else ["country", "source_id", "status"]
])
print("Successful:", int(download_results.status.eq("ok").sum()), "/ 100")
print("Failures remain in download_manifest.csv; no sources were silently replaced.")


In [ ]:
import matplotlib.pyplot as plt
import zipfile

manifest = pd.read_csv(ROOT / "download_manifest.csv")
good = manifest[manifest.status.eq("ok")]
if good.empty:
    raise RuntimeError("No usable chips. Inspect download_manifest.csv before continuing.")

for path in good.chip_file:
    validate_chip(ROOT / path)

fig, axes = plt.subplots(3, 3, figsize=(11, 10), squeeze=False)
for row_axes, row in zip(axes, good.head(3).itertuples(index=False)):
    with np.load(ROOT / row.chip_file, allow_pickle=False) as data:
        rgb = data["reflectance"][[2, 1, 0]].transpose(1, 2, 0)
        row_axes[0].imshow(np.nan_to_num(np.clip(rgb / 0.3, 0, 1), nan=0))
        row_axes[1].imshow(data["valid_mask"], vmin=0, vmax=1, cmap="gray")
        row_axes[2].imshow(data["worldcover"], vmin=0, vmax=100, cmap="tab20")
    row_axes[0].set_title(f"{row.country}: RGB")
    row_axes[1].set_title("Clear/valid mask")
    row_axes[2].set_title("WorldCover class IDs")
for axis in axes.flat:
    axis.axis("off")
plt.tight_layout()
plt.savefig(ROOT / "qa_preview.png", dpi=150)
plt.show()

bundle = ROOT.parent / "nb4_sih_cv_pilot.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ROOT.rglob("*")):
        if path.is_file():
            archive.write(path, arcname=str(path.relative_to(ROOT.parent)))
print("Download:", bundle)
print("ZIP size (MiB):", round(bundle.stat().st_size / 2**20, 1))
print("Single-scene acquisition pilot only. No model has been trained.")
